# **Scraping Planos Vivo**

_Página de planos do provedor: https://vivo.com.br/para-voce/produtos-e-servicos/para-casa/internet_

## **2 Tratamento**

### **2.1 Lista de códigos de produtos/promoções exibidas na página**

In [1]:
from bs4 import BeautifulSoup
from urllib.parse import parse_qs
import html

soup = BeautifulSoup(open("../../data/scraped/vivo.html", encoding="utf-8"), "html.parser")
planos_na_pagina = list({
    f'{q.get("productsIds", [""])[0]}|@|{q.get("promotion", [""])[0]}'
    for a in soup.find_all("a", href=True)
    if "vivofibra" in a["href"]
    for q in [parse_qs(html.unescape(a["href"]).split("?")[1])]
})
print(planos_na_pagina)

['106|@|GAMER', '317,327|@|', '2|@|Pro_160', '122|@|', '235|@|600', '235|@|Exitlag', '317,328|@|Fibra100', '317|@|', '235,266|@|600', '127|@|', '235,328|@|Fibra100', '106|@|', '317,329|@|Fibra100', '105|@|Fibra_120', '235,327|@|', '235,267|@|600', '235,329|@|Fibra100', '105|@|GAMER', '235,265|@|600', '235,264|@|600']


### **2.2 Filtro: apenas planos na página**

In [2]:
import pandas as pd
df = pd.read_json("../../data/scraped/vivo.json")
df = df[
    (
        df["codigoDaOferta"].fillna("").astype(str)
        + "|@|"
        + df["codigoPromocional"].fillna("").astype(str)
    ).isin(planos_na_pagina)
]
df.head()

ValueError: If using all scalar values, you must pass an index

### **2.3 Tratamento das colunas de interesse**

In [ ]:
from datetime import date
import re

# Data e provedor
df["DATA"] = date.today().strftime("%Y-%m-%d")
df["ISP"] = "Vivo"

# Velocidade
df["VELOCIDADE"] = df["textoFibra"].str.extract(r"(\d+)").astype(float)
df.loc[
    df["textoFibra"].str.contains(r"giga|gb|g\b", case=False, na=False, regex=True),
    "VELOCIDADE"
] *= 1000

# SVA: cria uma lista as colunas que iniciam com "cortesiaFibra" ou "sva" ou "textoTv"
# depois concatena essas colunas
cols_sva = [c for c in df if c.startswith(("cortesiaFibra", "sva", "textoTv"))]

df["SVAs"] = df[cols_sva].apply(
    lambda row: ", ".join(row.dropna().astype(str)),
    axis=1
)

# Preço
df["PRECO"] = (
    df["total"]
    .str.replace(r"[^\d,]", "", regex=True)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "VIVO.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")